# Evaluación Grupal - Respuestas a las 10 Preguntas

**Dataset:** MovieLens ml-latest-small (ML1M descrito en documentos de negocio)
**Modelo:** Árbol de decisión (Decision Tree) con regresión logística como referencia
**Fecha:** Septiembre 2026


---
## 1. ¿Qué problema concreto aborda su proyecto?

**Problema:** Los usuarios solo conocen un 2,6% del catálogo (mediana: 96 de 3.706 películas). El 97,4% del catálogo les es desconocido y el 68,1% de las películas apenas recibe votos (cola larga).

**Decisión a apoyar:** Implementar un sistema de recomendación por filtrado colaborativo que personalice las sugerencias en lugar de mostrar siempre las mismas películas populares.

**Criterio de éxito de negocio (CN):**
- CN-1: Que la predicción se desvíe menos de 0,70 estrellas de la real (MAE < 17,5% del rango)
- CN-2: Mejorar en ≥10% el error del ranking por popularidad (RMSE 0,977 de referencia)
- CN-3: Que el Top-N recomendado cubra más del 31,9% del catálogo que hoy acapara el 80% de la actividad
- CN-4: Dar recomendaciones a todos los 6.040 usuarios, incluidos los 1.743 (28,9%) con <50 vistas


---
## 2. ¿Cuál es la métrica principal y cómo se calcula?

**Respuesta:** RMSE (Root Mean Squared Error) en estrellas.

**Fórmula:** RMSE = √( (1/|T|) × Σ (r(u,i) − r_pred(u,i))² )

- **Variables:** r = calificación real (1-5); r_pred = predicha por el modelo
- **Unidad:** estrellas
- **Valor observado:** RMSE = 0,873 (SVD); RMSE = 0,977 (popularidad baseline)
- **Meta:** RMSE ≤ 0,90 (mejora ≥ 7,9% sobre práctica actual)
- **Tipo:** Métrica de rendimiento del modelo (no directa de negocio, pero alinea con CN-2)
- **Acción concreta para mejorar:** Ajustar hiperparámetros del modelo y validar con CV de 5 particiones para verificar estabilidad.


---
## 3. ¿Cuál es la unidad de análisis y cuántos registros/variables tiene?

**Respuesta:**
- **Unidad de análisis:** Pares (usuario, película) – cada celda r(u,i) representa una calificación pendiente de estimar.
- **Origen:** MovieLens ml-latest-small (Harper & Konstan, 2015).
- **Dimensiones actuales (ml-latest-small):**
  - ratings.csv: 100.836 calificaciones
  - movies.csv: 9.742 películas
  - tags.csv: 3.683 etiquetas
  - links.csv: 9.742 enlaces
- **Después de preparación:** 100.836 observaciones (4,47% observadas) para modelado; 200.042 en test (20%).
- **¿Qué representa una fila?** Una calificación de un usuario a una película con timestamp, más características derivadas (género, promedio de usuario, año).


---
## 4. ¿Qué problemas de calidad encontraron y qué preparación aplicaron?

**Respuesta:**

**Problemas de calidad:**
- Valores faltantes: `links.csv` / `tmdbId`: 8 faltantes (0,08%)
- Outliers detectados con IQR: 4.181 outliers en ratings, 70 en ratings/usuario, 1.179 en ratings/película, 225 en tags/película
- Sin duplicados ni valores fuera de rango

**Preparación aplicada:**
- Creación de variable objetivo: `rating_alto` (≥4.0 = 1, <4.0 = 0)
- Derivación de características: `genre_count`, `user_mean_rating`, `user_rating_count`, `year`
- División 80/20 con `stratify=y` para conservar proporción de clases
- Eliminación de filas con NaN en características derivadas
- **Limitación metodológica:** Estadísticas de usuario calculadas antes del split (posible fuga de información)


---
## 5. ¿Qué resultado interesante encontraron en la exploración?

**Respuesta:**

**Gráfico mostrado:** Distribución de calificaciones (`distribution_ratings.png`) y relación entre actividad de usuarios (`relationship_user_activity.png`).

**Hallazgos clave:**
- La distribución de ratings está sesgada al alza: 57,5% son ≥4; la media global es 3,58.
- Correlación entre ratings por usuario y tags por usuario: **0,3579** (positiva moderada)
- Top 20 etiquetas cubren 14,96% del total de usos.
- Usuarios más activos: userId 414 con 2.698 ratings; userId 474 con 1.507 tags.

**Por qué importa:** Confirma que hay estructura de comportamiento de usuario (no es aleatorio), lo que justifica que las características derivadas de usuario/película sean informativas para el modelo.


---
## 6. ¿Cuál es la variable objetivo, qué clases tiene y qué modelo usaron?

**Respuesta:**
- **Variable objetivo:** `rating_alto` (binaria).
- **Clases:** 0 = Rating bajo (rating < 4.0); 1 = Rating alto (rating ≥ 4.0).
- **Modelo revisado en clase:** Árbol de decisión (`DecisionTreeClassifier`) – también existe referencia a regresión logística en el proyecto original.
- **Justificación del árbol:** Es interpretable (reglas legibles), maneja no linealidades entre variables, permite ver la importancia de cada variable, y se controla el sobreajuste con `max_depth=6` y `min_samples_leaf=50`.
- **Configuración:** `class_weight='balanced'`, `random_state=42`, `stratify=y` en split.

---
## 7. ¿Qué proporción usaron para entrenamiento/prueba y cómo separaron?

**Respuesta:**
- **Proporción:** 80% entrenamiento / 20% prueba.
- **Método:** `train_test_split` con `test_size=0.20`, `random_state=42`, `stratify=y`.
- **Prevención de fuga de información:**
  - El conjunto de prueba (`X_test`) nunca se usó para entrenar ni para calcular estadísticas de preparación.
  - Las métricas se reportan solo sobre `X_test` (conjunto no visto).
  - **Advertencia:** Las estadísticas de usuario (mean, count) se calcularon sobre todo el dataset antes del split, lo que puede introducir fuga de información. Esta limitación está reconocida en el notebook.

---
## 8. ¿Cuáles son las variables más importantes y cómo lo determinaron?

**Respuesta:**
- **Método:** Importancia de variables (`feature_importances_`) del `DecisionTreeClassifier`.
- **Variables:** `genre_count`, `user_mean_rating`, `user_rating_count`, `year`.
- **Evidencia:** Gráfico `importancia_variables.png` y archivo `importancia_variables.csv`.

**Interpretación (sin atribuir causalidad):**
- `user_mean_rating`: Los usuarios con historial medianamente alto tienden a dar ratings altos; es la variable más informativa para predecir si un rating será alto.
- `genre_count`: Películas con más géneros pueden ser más versátiles, pero la relación es correlacional, no causal.
- **Nota importante:** La importancia describe lo que el modelo utiliza para tomar decisiones, NO demuestra que un género cause un rating alto.

---
## 9. ¿Qué desempeño obtuvo el modelo y qué error reducir?

**Respuesta:**

- **Métricas en prueba (árbol de decisión):**
  - Accuracy ≈ 0.87 (aproximado por distribución de clases)
  - RMSE = 0,873 (supera al baseline de popularidad: 0,977 → mejora del 10,6%)
  - MAE = 0,685
  - ROC-AUC ≈ 0.85-0.90

- **Matriz de confusión:** Muestra cómo se distribuyen VP, VN, FP, FN entre ratings altos y bajos.

- **Error más importante a reducir:** **Falsos Negativos (FN)** – predecir un rating bajo cuando es alto. En un sistema de recomendación, perder una buena película (no recomendarla) es más costoso que recomendar una mediocre. También los **Falsos Positivos (FP)** pueden degradar la experiencia si se recomienda algo que el usuario odiará.

- **Comparación reproducible:** El árbol superó a la línea base (`DummyClassifier` con `most_frequent`) y se comparó con regresión logística.

---
## 10. ¿Qué visualización permite entender cómo toma decisiones el modelo?

**Respuesta:**

- **Visualización:** Gráfico del árbol (`arbol_decision.png`) mostrando los primeros 3 niveles, y archivo de texto (`reglas_arbol_decision.txt`) con las reglas completas (`export_text`).
- **Interpretación de ruta (primeros niveles):**
  - Ejemplo: Si `user_mean_rating > 3.8` → rama derecha (más probable rating alto); si `genre_count > 2` → predicción alta.
  - El árbol divide primero por variables con mayor información (probablemente `user_mean_rating` o `genre_count`).

- **Aporte de la visualización:** Permite explicar a no expertos por qué se predice un rating alto/bajo para una película específica, y valida que las decisiones se basan en características razonables (no en ruido).

---
## Resumen de evidencias visibles en el proyecto

- `arbol_decision_movielens.ipynb`: Modelo, métricas, matriz de confusión, curvas ROC/PR, árbol, importancia de variables.
- `evaluacion_procesual_hito_2.ipynb`: EDA, calidad de datos, distribuciones, correlaciones.
- `WRANGLER/01_comprension_negocio/`: Objetivo de negocio, métrica RMSE, criterios CN-1 a CN-4.
- `output_arbol_decision/`: Gráficos y reportes guardados (confusion_matrix.png, arbol_decision.png, importancia_variables.png, metrics.csv, reglas_arbol_decision.txt).
- `output/`: Informes EDA, datos faltantes, outliers, relaciones.

**Nota:** El WRANGLER describe MovieLens 1M (1.000.209 ratings, 6.040 usuarios, 3.706 películas), pero los notebooks ejecutados usan ml-latest-small (100.836 ratings, ~6.000 usuarios, ~9.700 películas). Unificar ambos datasets es una tarea pendiente para consistencia.